<div id="singlestore-header" style="display: flex; background-color: rgba(235, 249, 245, 0.25); padding: 5px;">
    <div id="icon-image" style="width: 90px; height: 90px;">
        <img width="100%" height="100%" src="https://raw.githubusercontent.com/singlestore-labs/spaces-notebooks/master/common/images/header-icons/database.png" />
    </div>
    <div id="text" style="padding: 5px; margin-left: 10px;">
        <div id="badge" style="display: inline-block; background-color: rgba(0, 0, 0, 0.15); border-radius: 4px; padding: 4px 8px; align-items: center; margin-top: 6px; margin-bottom: -2px; font-size: 80%">SingleStore Notebooks</div>
        <h1 style="font-weight: 500; margin: 8px 0 0 4px;">Scale Read Workloads with Smart Attach Read Replicas</h1>
    </div>
</div>

## The story

It's the last week of March at **AdVenture Inc.**, the New York real-time ad startup where you are still the only developer. In February, resource pools fenced Theo's campaign reports into 20% of the CPU, and ad serving has been fast ever since.

Then the quarter closed. Theo needs a week of reports in two days, Finance closes the books on live spend, and Morgan Blaze, the CMO, has promised advertisers an AI agent that answers questions about their own campaigns. A pool can only divide one cluster's CPUs. This quarter, the readers need machines of their own.

**Smart Attach** attaches one database to more than one cluster: one cluster reads and writes it, and any number of others attach it **read-only**. The readers copy nothing. They read the same database from shared object storage, keep their own cache, and replay the writer's changes as they happen.

By 8 AM there are five tickets in your queue:

| Ticket | Problem | What you will do and measure |
|---|---|---|
| ADV-341, Analytics | Quarter-end reports need more CPU than a pool can spare from ad serving | Create a reader cluster, attach the database read-only, and measure ad-serving latency with reports on the writer, then on the reader |
| ADV-342, Morgan | The AI agent must be *unable* to change anything | Try every kind of write on the reader |
| ADV-343, Finance | How old are the numbers the reader returns? | Time how long a budget charge on the writer takes to appear on the reader, with the writer idle and busy |
| ADV-344, Data Engineering | The reader should be sized on its own, and schema changes should reach it | Add a column on the writer and watch it arrive; compare the two clusters' sizes |
| ADV-345, Morgan | Turn the reader off after the close, without touching ad serving | Detach the database from the reader while the writer keeps writing |

**Lab notes** boxes record what we measured across repeated runs.

## What's in this notebook:

1. Load the ad platform's data on this cluster, the writer.
2. Give the reports a reader of their own.
3. Measure ad serving with reports on the writer, then on the reader.
4. Make sure the AI agent cannot write.
5. Measure how fresh the reader is.
6. Watch a schema change reach the reader, and size the reader on its own.
7. Detach the reader while ads keep serving.
8. Clean up.
9. Wrap up: what Smart Attach is and when to use it.

### Before you start

- Run this notebook on an **Enterprise** edition cluster. Read-only attachments need a second cluster, which a Free Starter Workspace cannot have.
- The notebook needs **a second cluster in the same region** to be the reader. Section 2 creates the smallest size, in this cluster's project, and section 7 terminates it; it is billed while it runs, usually under half an hour. If your account cannot create another cluster, set `READER_CLUSTER` in section 2 to the name of one you already have in this region, and the notebook will attach to it, read from it and detach again, without resizing or terminating it.

## Questions?

Reach out to us through our [forum](https://www.singlestore.com/forum).

Creating a cluster from a notebook needs version 1.18 or later of the `singlestoredb` client. Install it, then **restart the kernel** (Kernel > Restart) before you run the next cell.

In [1]:
%pip install -q "singlestoredb>=1.18"

In [2]:
import singlestoredb as s2

if not hasattr(s2, 'manage_clusters'):
    raise RuntimeError(f'singlestoredb {s2.__version__} is loaded. Restart the kernel (Kernel > Restart), '
                       'then run this cell again.')
print(f'singlestoredb {s2.__version__}')

<div class="alert alert-block alert-warning">
    <b class="fa fa-solid fa-exclamation-circle"></b>
    <div>
        <p><b>Action Required</b></p>
        <p>Leave the database drop-down at the top of this notebook as it is. The next cell creates <tt>ad_platform</tt> on this cluster and switches to it.</p>
    </div>
</div>

Create the database. The cell keeps an existing `ad_platform` instead of dropping it, because dropping the database a notebook session is using leaves that session with no database. Section 1 clears out its tables anyway.

In [3]:
shared_tier_check = %sql show variables like 'is_shared_tier'
if shared_tier_check and shared_tier_check[0][1] == 'ON':
    raise RuntimeError('This is a Free Starter Workspace. Read-only attachments need an Enterprise '
                       'edition cluster, so this notebook cannot run here.')

%sql CREATE DATABASE IF NOT EXISTS ad_platform;
%sql USE ad_platform;
DATABASE = 'ad_platform'
print(f'Using database {DATABASE}')

## 1. Load the ad platform's data on this cluster, the writer.

Start with a fresh copy of AdVenture's platform, the same data as the previous notebook in this series, [Tune a Real-Time Ad Platform](https://www.singlestore.com/spaces/tune-a-real-time-ad-platform-with-resource-pools-returning-and-projections/). This cluster is the **writer**: ad serving and budget charges stay here.

The data is SingleStore's public [real-time digital marketing demo](https://github.com/singlestore-labs/demo-realtime-digital-marketing): 3 million ad requests and 500 thousand purchases, streamed from S3 with pipelines into five tables:

- `requests` and `purchases` -- the raw event streams, sharded by subscriber.
- `offers` -- one campaign per advertiser, with a maximum bid, a budget and spend so far.
- `notifications` -- every ad shown: when, to whom, which offer, and what it cost. Reports read this table.
- `subscriber_segments` -- audience memberships such as `buyer:Photobug`, each with an expiry time.

The load derives timestamps, 356 offers (one per vendor) and 3.5 million segment memberships from the streams. It uses `CRC32`, not `RAND()`, so every run produces the same data.

In [4]:
import json
import random
import statistics
import threading
import time

import singlestoredb as s2


def connect():
    """Open a connection to the notebook's database.

    New connections start in whatever database the drop-down menu selects,
    which may be none, so select ``DATABASE`` explicitly.

    Returns
    -------
    Connection
        An open connection with ``DATABASE`` selected.

    """
    conn = s2.connect()
    with conn.cursor() as cur:
        cur.execute(f'USE {DATABASE}')
    return conn


def query(sql, args=None):
    """Run one statement on a fresh connection and return its rows.

    Parameters
    ----------
    sql : str
        The statement to run.
    args : tuple, optional
        Parameters for the statement.

    Returns
    -------
    list of tuple
        The result rows, or an empty list for statements without a result set.

    """
    with connect() as conn, conn.cursor() as cur:
        cur.execute(sql, args)
        return list(cur.fetchall()) if cur.description else []

The notebook is safe to run more than once. If you ran the previous notebook and kept its data, its projection on `notifications` blocks `DROP TABLE`, so the next cell removes it first.

In [5]:
if query("SELECT 1 FROM information_schema.tables WHERE table_schema = DATABASE() AND table_name = 'notifications'"):
    query('DROP PROJECTION IF EXISTS notifications_by_offer ON notifications')

In [6]:
%%sql
DROP PIPELINE IF EXISTS requests_pipeline;
DROP PIPELINE IF EXISTS purchases_pipeline;
DROP TABLE IF EXISTS requests;
DROP TABLE IF EXISTS purchases;
DROP TABLE IF EXISTS offers;
DROP TABLE IF EXISTS notifications;
DROP TABLE IF EXISTS subscriber_segments;

CREATE TABLE requests (
    subscriber_id BIGINT NOT NULL,
    domain TEXT NOT NULL,
    ts DATETIME(6) NOT NULL,
    SORT KEY (ts),
    SHARD KEY (subscriber_id)
);

CREATE TABLE purchases (
    subscriber_id BIGINT NOT NULL,
    vendor TEXT NOT NULL,
    ts DATETIME(6) NOT NULL,
    SORT KEY (ts),
    SHARD KEY (subscriber_id)
);

CREATE TABLE offers (
    offer_id BIGINT NOT NULL,
    customer TEXT NOT NULL,
    enabled BOOLEAN NOT NULL DEFAULT TRUE,
    maximum_bid_cents BIGINT NOT NULL,
    budget_cents BIGINT NOT NULL,
    spend_cents BIGINT NOT NULL DEFAULT 0,
    PRIMARY KEY (offer_id),
    SHARD KEY (offer_id),
    SORT KEY (offer_id)
);

CREATE TABLE notifications (
    ts DATETIME(6) NOT NULL,
    subscriber_id BIGINT NOT NULL,
    offer_id BIGINT NOT NULL,
    cost_cents BIGINT NOT NULL,
    SORT KEY (ts),
    SHARD KEY (subscriber_id)
);

CREATE TABLE subscriber_segments (
    subscriber_id BIGINT NOT NULL,
    segment TEXT NOT NULL,
    expires_at DATETIME(6) NOT NULL,
    SORT KEY (expires_at),
    SHARD KEY (subscriber_id)
);

Each event gets a timestamp in the last 30 days. A pipeline's `SET` clause can only compute from `@variables`, not target columns, so both fields are read into variables first. The load takes a few minutes.

In [7]:
%%sql
CREATE PIPELINE requests_pipeline AS
    LOAD DATA S3 'singlestore-realtime-digital-marketing/v2/1m-16p/requests.*'
    CONFIG '{\"region\": \"us-east-1\"}'
    CREDENTIALS '{}'
    INTO TABLE requests FORMAT PARQUET (@sub <- subscriberid, @domain <- domain)
    SET subscriber_id = @sub, domain = @domain,
        ts = NOW(6) - INTERVAL (CRC32(CONCAT(@sub, @domain)) % 2592000) SECOND;

CREATE PIPELINE purchases_pipeline AS
    LOAD DATA S3 'singlestore-realtime-digital-marketing/v2/1m-16p/purchases.*'
    CONFIG '{\"region\": \"us-east-1\"}'
    CREDENTIALS '{}'
    INTO TABLE purchases FORMAT PARQUET (@sub <- subscriberid, @vendor <- vendor)
    SET subscriber_id = @sub, vendor = @vendor,
        ts = NOW(6) - INTERVAL (CRC32(CONCAT(@sub, @vendor)) % 2592000) SECOND;

START PIPELINE requests_pipeline FOREGROUND;
START PIPELINE purchases_pipeline FOREGROUND;

Now derive the rest. Every ad request becomes a shown ad, charged at its offer's maximum bid. Subscribers stay in `visits:<domain>` for 28 days after their last visit and in `buyer:<vendor>` for 29 days after their last purchase, so a few percent of memberships have already lapsed.

In [8]:
%%sql
INSERT INTO offers (offer_id, customer, maximum_bid_cents, budget_cents)
SELECT ROW_NUMBER() OVER (ORDER BY vendor), vendor,
       5 + CRC32(vendor) % 46,
       100 * (500 + CRC32(REVERSE(vendor)) % 4500)
FROM (SELECT DISTINCT vendor FROM purchases) v;

INSERT INTO notifications (ts, subscriber_id, offer_id, cost_cents)
SELECT r.ts, r.subscriber_id, o.offer_id, o.maximum_bid_cents
FROM (SELECT ts, subscriber_id, 1 + CRC32(CONCAT(subscriber_id, domain)) % 356 AS offer_id FROM requests) r
JOIN offers o ON o.offer_id = r.offer_id;

INSERT INTO subscriber_segments (subscriber_id, segment, expires_at)
SELECT subscriber_id, CONCAT('visits:', domain), MAX(ts) + INTERVAL 28 DAY FROM requests GROUP BY 1, 2
UNION ALL
SELECT subscriber_id, CONCAT('buyer:', vendor), MAX(ts) + INTERVAL 29 DAY FROM purchases GROUP BY 1, 2;

UPDATE offers o
JOIN (SELECT offer_id, SUM(cost_cents) AS s FROM notifications GROUP BY 1) n ON o.offer_id = n.offer_id
SET o.spend_cents = LEAST(n.s, o.budget_cents);

Loading in small batches leaves many small segments, and the background merger that combines them competes for CPU while it runs. Merge them now so the measurements below are steady:

In [9]:
%%sql
OPTIMIZE TABLE requests FULL;
OPTIMIZE TABLE purchases FULL;
OPTIMIZE TABLE notifications FULL;
OPTIMIZE TABLE subscriber_segments FULL;

In [10]:
%%sql
SELECT 'requests' AS table_name, COUNT(*) AS row_count FROM requests
UNION ALL SELECT 'purchases', COUNT(*) FROM purchases
UNION ALL SELECT 'offers', COUNT(*) FROM offers
UNION ALL SELECT 'notifications', COUNT(*) FROM notifications
UNION ALL SELECT 'subscriber_segments', COUNT(*) FROM subscriber_segments;

## 2. Give the reports a reader of their own.

<div style="border: 1px solid #f0a35e; border-radius: 6px; background: rgba(240,163,94,0.08); padding: 10px 14px; margin: 8px 0;">
<span style="font-family: monospace; opacity: 0.8">ADV-341</span> &nbsp;<span style="opacity:0.7">opened 8:05 AM</span><br/>
<b>Theo Park, Analytics:</b> Close starts today. The <tt>campaign_reports</tt> pool kept ads fast in February, but at 20% of the CPU my close reports will take all week. Can I have a machine of my own? And no copies: the last time someone exported a CSV, Finance closed the quarter on stale numbers.
</div>

A pool divides one cluster's CPUs; it cannot add more. A **reader** is a second cluster with its own CPUs, memory and cache, attached to the same database **read-only**. Nothing is copied: the database stays in shared object storage, the reader caches what its queries touch, and it replays the writer's changes as they are committed.

The reader must be in the **same region** as the writer. The next cell creates the smallest cluster in this cluster's region and project, with this cluster's firewall rules, and waits until it is running, which takes a few minutes. It expires after 4 hours in case the notebook stops before section 7 terminates it.

To use a cluster you already have instead, set `READER_CLUSTER` to its name. The notebook never resizes or terminates a cluster it did not create.

In [11]:
import os
from urllib.parse import unquote, urlparse

# None creates a new reader. To use an existing cluster in the same region, set its name here.
READER_CLUSTER = None
READER_SIZE = 'S-00'

clusters = s2.manage_clusters()
writer = clusters.get_cluster(os.environ['SINGLESTOREDB_WORKSPACE'])
print(f'Writer: {writer.name}, size {writer.size}, {writer.provider} {writer.region.region_name}')

In [12]:
def find_cluster(name):
    """Return the running cluster called ``name`` in this organization, or None."""
    matches = [c for c in clusters.clusters if c.name == name and c.state != 'TERMINATED']
    return matches[0] if matches else None


created_reader = False
if READER_CLUSTER:
    reader = find_cluster(READER_CLUSTER)
    if reader is None:
        raise RuntimeError(f'No cluster named {READER_CLUSTER!r} was found.')
else:
    # A rerun finds the reader an earlier run created, instead of making a second one.
    name = f'{writer.name}-reports'
    reader = find_cluster(name)
    if reader is None:
        try:
            reader = clusters.create_cluster(
                name=name, region=writer.region, size=READER_SIZE, project=writer.project,
                firewall_ranges=list(writer.firewall_ranges), allow_all_traffic=writer.allow_all_traffic,
                expires_at='4h', wait_on_active=True, wait_timeout=1800)
        except s2.ManagementError as e:
            if 'maximum number' in str(e):
                raise RuntimeError('Your account cannot create another cluster. Set READER_CLUSTER '
                                   'to the name of an existing cluster in the same region, and run '
                                   'this cell again.') from e
            raise
    created_reader = True

if reader.id == writer.id:
    raise RuntimeError('The reader must be a different cluster from the one running this notebook.')
if (reader.provider, reader.region.region_name) != (writer.provider, writer.region.region_name):
    raise RuntimeError(f'{reader.name} is in {reader.provider} {reader.region.region_name}; '
                       'a reader must be in the same region as the writer.')
if reader.state != 'ACTIVE':
    raise RuntimeError(f'{reader.name} is {reader.state}. Resume it in the portal, then run this cell again.')

print(f'Reader: {reader.name}, size {reader.size}' + (' (created by this notebook)' if created_reader else ''))

**Connect to the reader as yourself.** The users you manage in the portal exist on every cluster in your organization, so the notebook connects to the reader with its own login and needs no password. That login is a token which expires after about an hour, so `reader_connect()` reads the current one from the environment every time.

In [13]:
def reader_connect(database=True):
    """Open a connection to the reader as the notebook's own user.

    Parameters
    ----------
    database : bool, optional
        Select ``DATABASE`` on the new connection.

    Returns
    -------
    Connection
        An open connection to the reader.

    """
    login = urlparse('singlestoredb://' + os.environ['SINGLESTOREDB_URL'].split('://')[-1])
    conn = s2.connect(host=reader.endpoint, port=3306, user=unquote(login.username),
                      password=unquote(login.password))
    if database:
        with conn.cursor() as cur:
            cur.execute(f'USE {DATABASE}')
    return conn


def reader_query(sql, args=None):
    """Run one statement on the reader and return its rows."""
    with reader_connect() as conn, conn.cursor() as cur:
        cur.execute(sql, args)
        return list(cur.fetchall()) if cur.description else []


# A new cluster can take a minute to accept connections after it reports ACTIVE.
for attempt in range(18):
    try:
        with reader_connect(database=False) as conn, conn.cursor() as cur:
            cur.execute('SELECT CURRENT_USER()')
            print('Connected to the reader as', cur.fetchone()[0])
        break
    except s2.OperationalError:
        if attempt == 17:
            raise
        time.sleep(10)

**Attach the database read-only.** `ATTACH DATABASE ... READ ONLY` runs on the reader. It copies no data, so it takes seconds, not the minutes a restore would. A rerun finds the database already attached and skips this step.

In [14]:
with reader_connect(database=False) as conn, conn.cursor() as cur:
    cur.execute('SELECT 1 FROM information_schema.schemata WHERE schema_name = %s', (DATABASE,))
    if cur.fetchall():
        print(f'{DATABASE} is already attached to {reader.name}.')
    else:
        t0 = time.perf_counter()
        cur.execute(f'ATTACH DATABASE {DATABASE} READ ONLY')
        print(f'Attached {DATABASE} to {reader.name} in {time.perf_counter() - t0:.0f} s.')
    cur.execute("""SELECT site_name, attach_type FROM information_schema.mv_cloud_attached_databases
                   WHERE database_name = %s ORDER BY attach_type DESC""", (DATABASE,))
    for site, kind in cur.fetchall():
        print(f'  {site:40s} {kind}')

Every cluster attached to the database is listed: one `READWRITE` row for the writer, and a `READONLY` row for each reader. Now read from it. The first run of a report fetches what it needs from object storage into the reader's cache; repeat runs read the cache.

In [15]:
CLOSE_REPORT = """
    SELECT o.customer, COUNT(*) AS ads_shown, SUM(n.cost_cents) / 100 AS spend_dollars
    FROM notifications n JOIN offers o ON o.offer_id = n.offer_id
    GROUP BY o.customer ORDER BY spend_dollars DESC LIMIT 5"""

with reader_connect() as conn, conn.cursor() as cur:
    for run in range(3):
        t0 = time.perf_counter()
        cur.execute(CLOSE_REPORT)
        rows = cur.fetchall()
        print(f'Run {run + 1}: {(time.perf_counter() - t0) * 1000:5.0f} ms')

print()
for customer, shown, spend in rows:
    print(f'{customer:16s} {shown:6d} ads   ${spend:,.2f}')
print('\nSame counts on both clusters:',
      query('SELECT COUNT(*) FROM notifications') == reader_query('SELECT COUNT(*) FROM notifications'))

<div class="alert alert-block alert-info">
    <b class="fa fa-solid fa-flask"></b>
    <div>
        <p><b>Lab notes: attaching</b></p>

<p>Attaching the 10-million-row database took under half a minute the first time, and seconds when the reader attached it again after a detach. The first report on a fresh reader ran several times slower than its repeats, while the reader filled its cache.</p>
<p>Size a reader for its <b>working set</b>, the data its queries touch: if that does not fit in its cache, queries keep fetching from object storage.</p>

    </div>
</div>

## 3. Measure ad serving with reports on the writer, then on the reader.

The ad server's query is small: find the highest bid among the offers a subscriber is targeted by. Theo's close reports are heavy joins over millions of rows. The next cell serves ads on the writer for 30 seconds three times: alone, with four report sessions on the **writer**, and with the same four sessions on the **reader**.

In [16]:
SERVE = """
    SELECT o.offer_id, o.maximum_bid_cents
    FROM subscriber_segments s JOIN offers o ON CONCAT('buyer:', o.customer) = s.segment
    WHERE s.subscriber_id = %s AND s.expires_at > NOW(6) AND o.enabled
    ORDER BY o.maximum_bid_cents DESC LIMIT 1"""

REPORTS = [
    """SELECT p.vendor, COUNT(DISTINCT r.domain) AS domains, COUNT(*) AS pairs
       FROM requests r JOIN purchases p ON p.subscriber_id = r.subscriber_id
       GROUP BY p.vendor ORDER BY pairs DESC LIMIT 5""",
    """SELECT SUBSTRING_INDEX(domain, '.', -1) AS tld, COUNT(DISTINCT subscriber_id)
       FROM requests WHERE domain RLIKE '^[a-m].*(box|zz|oo).*' GROUP BY 1""",
]

CONNECT = {'writer': connect, 'reader': reader_connect}

buyers = [r[0] for r in query("SELECT subscriber_id FROM subscriber_segments WHERE segment LIKE 'buyer:%' LIMIT 2000")]

# Each cluster compiles a query shape on first use and caches the data it reads.
# Run every query once on both, so neither lands inside a measurement.
for where in CONNECT:
    with CONNECT[where]() as conn, conn.cursor() as cur:
        for sql in REPORTS:
            cur.execute(sql)
            cur.fetchall()
query(SERVE, (buyers[0],))


def run_reports(where, stop, done):
    """Run close reports on ``where`` until ``stop`` is set, appending 1 to ``done`` per report."""
    with CONNECT[where]() as conn, conn.cursor() as cur:
        i = 0
        while not stop.is_set():
            cur.execute(REPORTS[i % len(REPORTS)])
            cur.fetchall()
            done.append(1)
            i += 1


def serve_ads(reports_on=None, seconds=30, report_threads=4):
    """Serve ads on the writer for ``seconds`` while reports run on ``reports_on``.

    Parameters
    ----------
    reports_on : str, optional
        'writer' or 'reader'. With None, no reports run.
    seconds : float, optional
        How long to serve ads.
    report_threads : int, optional
        Number of concurrent report sessions.

    Returns
    -------
    dict
        Serving latency percentiles in milliseconds, and the number of reports completed.

    """
    stop, done = threading.Event(), []
    reporters = [threading.Thread(target=run_reports, args=(reports_on, stop, done))
                 for _ in range(report_threads if reports_on else 0)]
    for r in reporters:
        r.start()
    time.sleep(3 if reports_on else 0)
    latencies = []
    with connect() as conn, conn.cursor() as cur:
        start_reports = len(done)
        end = time.time() + seconds
        while time.time() < end:
            t0 = time.perf_counter()
            cur.execute(SERVE, (random.choice(buyers),))
            cur.fetchall()
            latencies.append((time.perf_counter() - t0) * 1000)
        reports = len(done) - start_reports
    stop.set()
    for r in reporters:
        r.join()
    q = statistics.quantiles(latencies, n=100)
    return {'p50': q[49], 'p95': q[94], 'reports': reports if reports_on else None}


results = {'Serving alone': serve_ads(),
           'Reports on the writer': serve_ads('writer'),
           'Reports on the reader': serve_ads('reader')}
base = results['Serving alone']
for label, r in results.items():
    reports = '' if r['reports'] is None else f"   reports completed: {r['reports']}"
    print(f"{label:24s} p50 {r['p50'] / base['p50']:4.1f}x   p95 {r['p95'] / base['p95']:4.1f}x{reports}")

Latency is shown as a multiple of serving alone. With the reports on the writer, they compete with ad serving for the same CPUs, and the slowest ads get two to four times slower. On the reader, they use other CPUs, and ad serving runs as if they were not there.

<div class="alert alert-block alert-info">
    <b class="fa fa-solid fa-flask"></b>
    <div>
        <p><b>Lab notes: isolation</b></p>

<p>Across three runs, ad-serving latency as a multiple of serving alone, with four report sessions:</p>
<table>
<tr><th></th><th>p50</th><th>p95</th></tr>
<tr><td>Reports on the writer</td><td>1.4-2.7x</td><td>1.9-4.5x</td></tr>
<tr><td>Reports on the reader</td><td>0.9-1.0x</td><td>1.0x</td></tr>
</table>
<p>On a reader the same size as the writer, the reports completed as many times as on the writer; on a reader twice the size, more than twice as many. Compare the previous notebook: a resource pool brought p95 down to about 2x by giving the reports less CPU. A reader brings it to 1x and gives the reports a whole cluster.</p>

    </div>
</div>

<div style="border-left: 4px solid #3cb371; background: rgba(60,179,113,0.08); padding: 8px 14px; margin: 8px 0;">
<span style="font-family: monospace">ADV-341</span> &nbsp;<b>Resolved.</b> Theo's close reports run on the reader, against the same database, with no copy to go stale. Ad serving did not notice.
</div>

## 4. Make sure the AI agent cannot write.

<div style="border: 1px solid #f0a35e; border-radius: 6px; background: rgba(240,163,94,0.08); padding: 10px 14px; margin: 8px 0;">
<span style="font-family: monospace; opacity: 0.8">ADV-342</span> &nbsp;<span style="opacity:0.7">opened 10:20 AM</span><br/>
<b>Morgan Blaze, Chief Marketing Officer:</b> The board loved it: advertisers will ask our AI agent about their own campaigns. Legal loved it less. The agent must be unable to change anything. Not "it doesn't have the permission". Unable.
</div>

In the previous notebook the agent got its own login with only `SELECT` grants. That still depends on nobody ever granting it more. A read-only attachment does not depend on grants: on the reader, the database itself refuses writes, for every user.

The notebook's own login can write to `ad_platform` on the writer. Point it at the reader and try every kind of change:

In [17]:
WRITES = [
    "INSERT INTO offers (offer_id, customer, maximum_bid_cents, budget_cents) VALUES (9999, 'Agent', 1, 1)",
    'UPDATE offers SET budget_cents = budget_cents * 2 WHERE offer_id = 42',
    'DELETE FROM notifications WHERE offer_id = 42',
    'TRUNCATE TABLE subscriber_segments',
    'CREATE TABLE agent_scratch (note TEXT)',
    'ALTER TABLE offers ADD COLUMN agent_note TEXT',
    'DROP TABLE purchases',
    'CREATE TEMPORARY TABLE agent_tmp (note TEXT)',
]

with reader_connect() as conn, conn.cursor() as cur:
    for sql in WRITES:
        try:
            cur.execute(sql)
            print(f'ALLOWED  {sql}')
        except s2.Error as e:
            print(f'refused  {e.errno}  {sql.split("(")[0][:60]}')
    cur.execute('SELECT COUNT(*) FROM offers WHERE offer_id = 42')
    print('\nReads still work:', cur.fetchone()[0], 'row for offer 42')

Every write fails with error **2719**: the database "is attached as a readonly database" on the reader "and is unavailable for write operations". That covers data changes and schema changes alike. Temporary tables fail too, with 1706, so an agent framework that stages results in a temporary table needs to run that step somewhere else.

Point the agent's connection string at the reader's endpoint, and the worst a bad prompt can do is a slow query on a cluster no customer is waiting on.

In [18]:
print('Agent endpoint (the reader):', reader.endpoint)

<div class="alert alert-block alert-info">
    <b class="fa fa-solid fa-flask"></b>
    <div>
        <p><b>Lab notes: read-only</b></p>

<p>Writes refused on the reader, for an admin login as for the notebook's: <tt>INSERT</tt>, <tt>UPDATE</tt>, <tt>DELETE</tt> and <tt>TRUNCATE</tt> (2719, reported by a leaf node), <tt>CREATE</tt>, <tt>ALTER</tt> and <tt>DROP</tt> (2719), and <tt>CREATE TEMPORARY TABLE</tt> (1706).</p>

    </div>
</div>

<div style="border-left: 4px solid #3cb371; background: rgba(60,179,113,0.08); padding: 8px 14px; margin: 8px 0;">
<span style="font-family: monospace">ADV-342</span> &nbsp;<b>Resolved.</b> The agent connects to the reader, where <tt>ad_platform</tt> cannot be changed by any login. Legal has its "unable".
</div>

## 5. Measure how fresh the reader is.

<div style="border: 1px solid #f0a35e; border-radius: 6px; background: rgba(240,163,94,0.08); padding: 10px 14px; margin: 8px 0;">
<span style="font-family: monospace; opacity: 0.8">ADV-343</span> &nbsp;<span style="opacity:0.7">opened 11:45 AM</span><br/>
<b>Priya Raman, Finance:</b> If Theo's reports run on a different machine, how old are their numbers? I close on spend charged up to the minute. "Near real time" is not a number I can put in an audit file.
</div>

The writer commits a change, then sends its transaction log to the readers asynchronously, and each reader replays it. Measure the gap the way Priya would see it: charge offer 42 on the writer with `UPDATE ... RETURNING`, which hands back the new balance, then read the reader until it shows that balance.

Measure twice: with the writer idle, then while 8 sessions serve ads and charge budgets on it.

In [19]:
def staleness(samples=12):
    """Charge offer 42 on the writer and time how long the reader takes to show each new balance.

    Parameters
    ----------
    samples : int, optional
        Number of charges to time.

    Returns
    -------
    list of float
        Seconds from each commit until the reader returned the new balance, sorted.

    """
    gaps = []
    with connect() as w, reader_connect() as r, w.cursor() as wc, r.cursor() as rc:
        for _ in range(samples):
            wc.execute('UPDATE offers SET spend_cents = spend_cents + 1 WHERE offer_id = 42 RETURNING spend_cents')
            balance = wc.fetchone()[0]
            t0 = time.perf_counter()
            while True:
                rc.execute('SELECT spend_cents FROM offers WHERE offer_id = 42')
                if rc.fetchone()[0] >= balance:
                    break
                time.sleep(0.05)
            gaps.append(time.perf_counter() - t0)
            time.sleep(0.5)
    return sorted(gaps)


def ad_traffic(stop, served):
    """Show ads and charge budgets on the writer until ``stop`` is set."""
    with connect() as conn, conn.cursor() as cur:
        while not stop.is_set():
            offer = random.randint(1, 356)
            cur.execute("""INSERT INTO notifications (ts, subscriber_id, offer_id, cost_cents)
                           SELECT NOW(6), %s, offer_id, maximum_bid_cents FROM offers WHERE offer_id = %s""",
                        (random.choice(buyers), offer))
            cur.execute('UPDATE offers SET spend_cents = spend_cents + 7 WHERE offer_id = %s', (offer,))
            served.append(1)


def show(label, gaps):
    print(f'{label:34s} min {gaps[0]:4.2f} s   median {gaps[len(gaps) // 2]:4.2f} s   max {gaps[-1]:4.2f} s')


show('Writer idle', staleness())

stop, served = threading.Event(), []
traffic = [threading.Thread(target=ad_traffic, args=(stop, served)) for _ in range(8)]
for t in traffic:
    t.start()
time.sleep(3)
t0, n0 = time.time(), len(served)
busy = staleness()
rate = (len(served) - n0) / (time.time() - t0)
stop.set()
for t in traffic:
    t.join()
show(f'Writer busy ({rate:,.0f} ads charged/s)', busy)

The gap includes the time to commit on the writer and to read on the reader, so it is an upper bound. SingleStore's objective for Smart Attach is that new data reaches read-only attachments within **1 minute**; here it took about a second or less.

<div class="alert alert-block alert-info">
    <b class="fa fa-solid fa-flask"></b>
    <div>
        <p><b>Lab notes: freshness</b></p>

<p>Across three runs of 12 to 15 charges each:</p>
<table>
<tr><th>Writer</th><th>Charge visible on the reader after</th></tr>
<tr><td>Idle</td><td>0.4 to 1.9 seconds, median 0.7 to 0.9</td></tr>
<tr><td>Busy, 50 to 370 ads charged per second</td><td>0.1 to 0.45 seconds, median about 0.3</td></tr>
</table>
<p>The reader was <b>fresher</b> while the writer was busy. A reader gives the same consistency as the writer, read committed; it only applies the writer's changes a little later. Each reader replays on its own, so two readers can differ by a moment.</p>

    </div>
</div>

<div style="border-left: 4px solid #3cb371; background: rgba(60,179,113,0.08); padding: 8px 14px; margin: 8px 0;">
<span style="font-family: monospace">ADV-343</span> &nbsp;<b>Resolved.</b> Priya's audit file says it: every charge reached the reports cluster in under two seconds, against an objective of one minute.
</div>

## 6. Watch a schema change reach the reader, and size the reader on its own.

<div style="border: 1px solid #f0a35e; border-radius: 6px; background: rgba(240,163,94,0.08); padding: 10px 14px; margin: 8px 0;">
<span style="font-family: monospace; opacity: 0.8">ADV-344</span> &nbsp;<span style="opacity:0.7">opened 1:30 PM</span><br/>
<b>Sam Whitfield, Data Engineering:</b> Two things. Finance wants a quarterly target on every offer, so I'm adding a column. Does Theo's cluster get it, or do I file a ticket with myself? And his cluster should be small most of the month and big at close, without anyone touching ad serving.
</div>

**Schema changes follow the data.** DDL runs on the writer and reaches readers through the same log as the data. Add Finance's column on the writer, then read it on the reader:

In [20]:
def has_target_column(on_reader=False):
    """Return True if offers has the quarter_target_cents column, on the writer or the reader."""
    sql = """SELECT 1 FROM information_schema.columns
             WHERE table_schema = %s AND table_name = 'offers' AND column_name = 'quarter_target_cents'"""
    return bool((reader_query if on_reader else query)(sql, (DATABASE,)))


if has_target_column():
    query('ALTER TABLE offers DROP COLUMN quarter_target_cents')
    while has_target_column(on_reader=True):
        time.sleep(0.2)

query('ALTER TABLE offers ADD COLUMN quarter_target_cents BIGINT NOT NULL DEFAULT 0')
t0 = time.perf_counter()
while not has_target_column(on_reader=True):
    time.sleep(0.1)
print(f'The new column reached the reader {time.perf_counter() - t0:.1f} s after the ALTER committed.')

query('UPDATE offers SET quarter_target_cents = budget_cents * 3 WHERE offer_id = 42')
time.sleep(2)
print('Offer 42 on the reader:', reader_query('SELECT customer, quarter_target_cents FROM offers WHERE offer_id = 42'))

Sam files no ticket: new tables and columns arrive on every reader without a step of their own.

**Size readers on their own.** The writer is sized for ad serving and charging; each reader is sized for its own queries. The one limit is that a reader needs at least one CPU core for every 8 partitions of the database:

In [21]:
reader.refresh()
print(f'Writer {writer.name}: {writer.size}')
print(f'Reader {reader.name}: {reader.size}')

For the close, Theo's reader can grow, and shrink again afterwards, through the Management API or the portal, because it is a cluster of its own. Set `RESIZE_READER = True` to try it on the reader this notebook created. The cell grows the reader one size, then shrinks it back, waiting for each change to finish.

In [22]:
RESIZE_READER = False

if RESIZE_READER and created_reader:
    for size in ['S-0', READER_SIZE]:
        t0 = time.time()
        reader.update(size=size, wait_on_active=True, wait_timeout=1800)
        print(f'Reader resized to {reader.size} in {time.time() - t0:.0f} s; '
              f'writer still {clusters.get_cluster(writer.id).size}')
elif RESIZE_READER:
    print(f'{reader.name} was not created by this notebook, so it is not resized.')

<div class="alert alert-block alert-info">
    <b class="fa fa-solid fa-flask"></b>
    <div>
        <p><b>Lab notes: schema changes</b></p>

<p>An added column was queryable on the reader 0.2 to 1.5 seconds after the <tt>ALTER</tt> committed, and a new table with its rows arrived as fast. Dropped columns disappear the same way.</p>

    </div>
</div>

<div style="border-left: 4px solid #3cb371; background: rgba(60,179,113,0.08); padding: 8px 14px; margin: 8px 0;">
<span style="font-family: monospace">ADV-344</span> &nbsp;<b>Resolved.</b> Finance's target column reached Theo's cluster on its own, and Theo's cluster has a size of its own.
</div>

## 7. Detach the reader while ads keep serving.

<div style="border: 1px solid #f0a35e; border-radius: 6px; background: rgba(240,163,94,0.08); padding: 10px 14px; margin: 8px 0;">
<span style="font-family: monospace; opacity: 0.8">ADV-345</span> &nbsp;<span style="opacity:0.7">opened 5:10 PM</span><br/>
<b>Morgan Blaze, Chief Marketing Officer:</b> The close is DONE. Turn off whatever we're paying for that we don't need until June. Ads keep running, obviously.
</div>

`DETACH DATABASE` on the reader removes its attachment and nothing else. The writer does not take part, so ad serving carries on. Detach while 8 sessions serve ads and charge budgets on the writer:

In [23]:
stop, served, errors = threading.Event(), [], []


def guarded_traffic():
    try:
        ad_traffic(stop, served)
    except Exception as e:
        errors.append(e)


traffic = [threading.Thread(target=guarded_traffic) for _ in range(8)]
for t in traffic:
    t.start()
time.sleep(3)

n0 = len(served)
with reader_connect(database=False) as conn, conn.cursor() as cur:
    t0 = time.perf_counter()
    cur.execute(f'DETACH DATABASE {DATABASE}')
    print(f'Detached {DATABASE} from {reader.name} in {time.perf_counter() - t0:.1f} s.')
time.sleep(5)
stop.set()
for t in traffic:
    t.join()
print(f'The writer charged {len(served) - n0} ads meanwhile, with {len(errors)} errors.')

try:
    reader_query('SELECT COUNT(*) FROM offers')
except s2.Error as e:
    print(f'On the reader now: {e.errno} {e.msg}')

Detaching keeps nothing on the reader and deletes nothing: the database lives on with its writer, and attaching it again later is the same one-line `ATTACH` as in section 2. A cluster with no work can then be suspended or terminated. If this notebook created the reader, terminate it now:

In [24]:
if created_reader:
    t0 = time.time()
    reader.terminate(wait_on_terminated=True, wait_timeout=1800)
    print(f'Terminated {reader.name} in {time.time() - t0:.0f} s.')
else:
    print(f'{reader.name} was not created by this notebook, so it keeps running.')

<div style="border-left: 4px solid #3cb371; background: rgba(60,179,113,0.08); padding: 8px 14px; margin: 8px 0;">
<span style="font-family: monospace">ADV-345</span> &nbsp;<b>Resolved.</b> The reports cluster is gone until June, and ad serving never noticed it leave. Zero open tickets.
</div>

## 8. Clean up.

Drop the database on the writer. If you used an existing cluster as the reader and stopped before section 7, the next cell detaches it first, so the drop leaves no reader holding the database.

In [25]:
if not created_reader:
    try:
        with reader_connect(database=False) as conn, conn.cursor() as cur:
            cur.execute('SELECT 1 FROM information_schema.schemata WHERE schema_name = %s', (DATABASE,))
            if cur.fetchall():
                cur.execute(f'DETACH DATABASE {DATABASE}')
                print(f'Detached {DATABASE} from {reader.name}.')
    except s2.Error as e:
        print(f'Could not check {reader.name}: {e}')

query(f'DROP DATABASE IF EXISTS {DATABASE}')
print(f'Dropped {DATABASE}.')

## 9. Wrap up.

Five tickets, one database, two clusters:

| Feature | What it is | Use it for | Watch out for |
|---|---|---|---|
| Read-only attachment | A second cluster attached to the same database with `ATTACH DATABASE ... READ ONLY`; nothing is copied | Reports, dashboards and AI agents that need live data and their own CPUs (ADV-341) | Same region as the writer; the first queries fill the reader's cache, so size it for its working set |
| Writes refused on the reader | Every `INSERT`, `UPDATE`, `DELETE` and DDL fails with error 2719, for every login | SQL you did not write, such as an AI agent's (ADV-342) | Temporary tables fail too (1706) |
| Replay from the writer's log | Readers apply the writer's committed changes about a second later, with the same read-committed consistency | Fresh numbers on a separate cluster (ADV-343) | Readers replay independently, so two readers can differ by a moment |
| DDL that follows the data | Tables and columns created on the writer appear on every reader | One schema to maintain (ADV-344) | Run all DDL on the writer |
| Independent size | Each reader is its own cluster, resized and paid for on its own | Sizing reporting capacity for the close separately from ad serving (ADV-344) | A reader needs at least one core per 8 partitions of the database |
| `DETACH DATABASE` on the reader | Removes the reader's attachment; the database and the writer carry on | Turning reporting capacity off between closes (ADV-345) | Suspend or terminate the idle cluster to stop paying for it |

<div id="singlestore-footer" style="background-color: rgba(194, 193, 199, 0.25); height:2px; margin-bottom:10px"></div>
<div><img src="https://raw.githubusercontent.com/singlestore-labs/spaces-notebooks/master/common/images/singlestore-logo-grey.png" style="padding: 0px; margin: 0px; height: 24px"/></div>